# experiment:olmoearth_zero_shot_cnn_448
## OlmoEarth frozen backbone → Shallow CNN Head (CE head, 224×224 output)

Uses `ShallowCNNHead`: 3 bilinear ×2 upsample stages, ~2.16M params.
Sits between the linear probe (98K) and UNet (11M) in capacity.
All utilities from `riverdecode/`.

## 0. Config

In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# ── EDIT THESE ───────────────────────────────────────────────────────────────
DATASET_DIR             = '/content/drive/MyDrive/CS682/project/RiverScope_dataset'
OUTPUT_DIR              = '/content/drive/MyDrive/CS682/project/results/olmoearth_zero_shot_cnn_448'
RIVERSCOPE_REPO_DIR     = '/content/drive/MyDrive/CS682/project/riverscope-models'
RESOLUTION_SUMMARY_PATH = '/content/drive/MyDrive/CS682/project/results/resolution_loss_analysis/resolution_loss_summary.json'
MODEL_SIZE       = 'OLMOEARTH_V1_BASE'
PATCH_SIZE       = 8
INPUT_SIZE       = 224          # encoder input resolution
H_PATCHES        = INPUT_SIZE // PATCH_SIZE   # 28
NUM_CLASSES      = 2
EPOCHS           = 50
BATCH_SIZE       = 32
RANDOM_SEED      = 42
DICE_WEIGHT      = 0.5
LR_SWEEP         = [1e-4, 5e-4, 1e-3, 5e-3, 1e-2]
MASK_KEY         = 'masks_224'  # CNN outputs 224×224 (patch_size=8 → 28×28→224)
# ─────────────────────────────────────────────────────────────────────────────

## 1. Environment Setup

In [ ]:
import sys, os
# ── Add RiverDecode to path so `riverdecode` is importable ──────────────────
RIVERDECODE_DIR = '/content/drive/MyDrive/CS682/project/riverscope-models/RiverDecode'
if RIVERDECODE_DIR not in sys.path:
    sys.path.insert(0, RIVERDECODE_DIR)

from riverdecode.setup import (
    mount_drive, patch_dynamo, setup_output_dirs, print_gpu_info,
    install_dependencies, clone_or_update_repo,
)
from riverdecode.data      import set_seeds, load_splits, parse_all_splits, load_resolution_ceiling, print_date_stats
from riverdecode.io        import read_image_for_model, read_image_for_display, read_mask_raw, verify_sample_tile
from riverdecode.backbone  import load_olmoearth, extract_spatial_tokens
from riverdecode.embeddings import load_or_compute_embeddings
from riverdecode.losses    import (
    combined_loss_bce, combined_loss_ce,
    compute_pos_weight, compute_f1_sigmoid, compute_f1_argmax,
    compute_iou_np, compute_f1_np, build_ce_criterion,
)
from riverdecode.training  import adjust_learning_rate, train_one_lr, SweepState, val_sanity_viz

In [ ]:
mount_drive()
install_dependencies()
clone_or_update_repo(RIVERSCOPE_REPO_DIR)
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print_gpu_info()
setup_output_dirs(OUTPUT_DIR)

## 2. Resolution Ceiling

In [ ]:
CEILING_IOU, CEILING_STD, CEILING_SSIM, CEILING_PSNR = load_resolution_ceiling(RESOLUTION_SUMMARY_PATH)

## 3. Dataset & Date Parsing

In [ ]:
set_seeds(RANDOM_SEED)
df_train, df_valid, df_test = load_splits(DATASET_DIR)
df_train, df_valid, df_test = parse_all_splits(df_train, df_valid, df_test)
verify_sample_tile(DATASET_DIR, df_test)
print_date_stats(df_train, df_valid, df_test)

## 4. Load OlmoEarth Backbone

In [ ]:
olmo_model, EMBED_DIM = load_olmoearth(MODEL_SIZE, DEVICE)

## 5. Embeddings

In [ ]:
# Tier 2: point to LP-224 embeddings if already computed (same INPUT_SIZE=224)
EXISTING_EMB_DIR = '/content/drive/MyDrive/CS682/project/results/olmoearth_zero_shot_linear_probe_paper_approach/embeddings'

for df, split in [(df_train, 'train'), (df_valid, 'valid'), (df_test, 'test')]:
    load_or_compute_embeddings(
        df, split, OUTPUT_DIR,
        existing_embeddings_dir=EXISTING_EMB_DIR,
        mask_key=MASK_KEY, h_patches=H_PATCHES, input_size=INPUT_SIZE,
        dataset_dir=DATASET_DIR, olmo_model=olmo_model,
        patch_size=PATCH_SIZE, device=DEVICE,
    )

## 6. DataLoaders & pos_weight

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

train_data = torch.load(f'{OUTPUT_DIR}/embeddings/train_embeddings.pt', map_location='cpu')
valid_data = torch.load(f'{OUTPUT_DIR}/embeddings/valid_embeddings.pt', map_location='cpu')

train_emb,  train_masks = train_data['embeddings'], train_data[MASK_KEY]
valid_emb,  valid_masks = valid_data['embeddings'], valid_data[MASK_KEY]
print(f'Train emb: {train_emb.shape}  masks: {train_masks.shape}')
print(f'Valid emb: {valid_emb.shape}  masks: {valid_masks.shape}')

train_loader = DataLoader(TensorDataset(train_emb, train_masks),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
valid_loader = DataLoader(TensorDataset(valid_emb, valid_masks),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

POS_WEIGHT = compute_pos_weight(f'{OUTPUT_DIR}/embeddings/train_embeddings.pt', mask_key=MASK_KEY)
criterion  = build_ce_criterion(POS_WEIGHT, DEVICE)

## 7. Model, Loss & Metric

In [ ]:
from riverdecode.models.cnn_head import ShallowCNNHead
import torch.nn as nn

# Sanity-check the architecture
_probe = ShallowCNNHead(EMBED_DIM, NUM_CLASSES)
_n     = sum(p.numel() for p in _probe.parameters())
_dummy = torch.zeros(2, H_PATCHES, H_PATCHES, EMBED_DIM)
_out   = _probe(_dummy)
print(f'ShallowCNNHead params  : {_n:,}')
print(f'Forward pass           : {list(_dummy.shape)} → {list(_out.shape)}')
assert _out.shape == (2, NUM_CLASSES, INPUT_SIZE, INPUT_SIZE), f'Shape mismatch: {_out.shape}'
print('Architecture verified.')
del _probe, _dummy, _out

def loss_fn(logits, masks):
    return combined_loss_ce(logits, masks, criterion, dice_weight=DICE_WEIGHT)

metric_fn = compute_f1_argmax

## 8. LR Sweep & Training

Paper default: select best by **last epoch** val F1 (not best epoch),
matching the linear probe protocol.

In [ ]:
sweep = SweepState()
for lr in LR_SWEEP:
    model, val_f1 = train_one_lr(
        lr,
        model_cls=ShallowCNNHead,
        model_kwargs=dict(embed_dim=EMBED_DIM, num_classes=NUM_CLASSES),
        train_loader=train_loader,
        valid_loader=valid_loader,
        output_dir=OUTPUT_DIR,
        loss_fn=loss_fn,
        metric_fn=metric_fn,
        total_epochs=EPOCHS,
        device=DEVICE,
    )
    sweep.update(lr, val_f1, model)

sweep.print_summary()
print(f'\nBest LR: {sweep.best_lr:.1e}  |  Val F1: {sweep.best_metric:.4f}')

## 9. Validation Visualisation

In [ ]:
val_sanity_viz(
    sweep.best_model, valid_data, OUTPUT_DIR,
    head_type='argmax', mask_key=MASK_KEY, lr=sweep.best_lr, device=DEVICE,
)

## 10. Test Evaluation

In [ ]:
import numpy as np, rasterio, skimage.transform as skt
from tqdm.auto import tqdm

best_model = sweep.best_model.eval().to(DEVICE)
test_data  = torch.load(f'{OUTPUT_DIR}/embeddings/test_embeddings.pt', map_location='cpu')
all_ious, all_f1s = [], []
pred_dir = os.path.join(OUTPUT_DIR, 'predictions')

with torch.no_grad():
    for i, emb in enumerate(tqdm(test_data['embeddings'], desc='Test')):
        logits = best_model(emb.unsqueeze(0).to(DEVICE))                 # [1, 2, 224, 224]
        prob   = torch.softmax(logits, dim=1)[0, 1].cpu().numpy()        # [224, 224]

        gt_path = test_data['gt_meta_paths'][i]
        gt_mask = read_mask_raw(gt_path)                                  # native 500×500

        # Upscale prediction to native GT resolution before metric calculation
        prob_native = skt.resize(prob, gt_mask.shape, order=1, anti_aliasing=True)

        all_ious.append(compute_iou_np(prob_native, gt_mask))
        all_f1s.append(compute_f1_np(prob_native,  gt_mask))

        # Save prediction as GeoTIFF with GT georeference
        with rasterio.open(gt_path) as src:
            meta = src.meta.copy()
        meta.update({'count': 1, 'dtype': 'uint8'})
        tile_id = test_data['tile_ids'][i]
        with rasterio.open(os.path.join(pred_dir, f'{tile_id}.tif'), 'w', **meta) as dst:
            dst.write((prob_native >= 0.5).astype('uint8')[np.newaxis])

print(f'Test IoU : {np.mean(all_ious):.4f} ± {np.std(all_ious):.4f}')
print(f'Test F1  : {np.mean(all_f1s):.4f} ± {np.std(all_f1s):.4f}')
print(f'Resolution ceiling IoU: {CEILING_IOU:.4f} ± {CEILING_STD:.4f}')